# AgentGuard Retrieval Quality Check

Use this notebook to inspect whether Elastic retrieves useful historical neighbors for a live-like trace.

In [ ]:
import json, os, pathlib, sys

repo_root = pathlib.Path.cwd()
sys.path.insert(0, str(repo_root / 'src'))

from agentguard.storage import AgentGuardElasticStore, load_elastic_config
from agentguard.tracing.schema_v1 import AgentGuardTraceV1

trace_path = repo_root / 'data' / 'traces' / 'v1' / 'openclaw' / 'traces.jsonl'
records = [json.loads(line) for line in trace_path.read_text().splitlines() if line.strip()]
trace = AgentGuardTraceV1.model_validate(records[-1])
trace.trace_id, trace.proposed_tool_call.tool_name, trace.retrieval_text.summary


In [ ]:
store = AgentGuardElasticStore(config=load_elastic_config())
response = store.search_similar_traces(trace, size=5)
[(hit.get('_score'), hit.get('_source', {}).get('trace_id'), hit.get('_source', {}).get('proposed_tool_call', {}).get('tool_name')) for hit in response.get('hits', {}).get('hits', [])]
